# Notebook 03 v2 — Probabilistic Forecasting and Scenario Generation

This notebook converts the frozen out-of-sample deterministic predictions from
Notebook 02 into calibrated probabilistic wind forecasts and joint DK1–DK2
scenarios.

Version 2 correctly permits missing predictions from sparse Notebook 02
baselines, audits them, and removes only those unusable rows before building
the probabilistic modelling dataset.


## Information-time design

For an evaluation fold \(f\), probabilistic calibration uses only residuals
from completed out-of-sample folds \(< f\).

Therefore:

- D2 fold 1 calibrates D2 fold 2;
- D2 folds 1–2 calibrate D2 fold 3;
- exact fold 1 calibrates exact fold 2;
- the first fold of each panel is reserved as calibration history and is not
  assigned probabilistic evaluation scores.

This avoids using future residuals or the current evaluation fold to estimate
its uncertainty distribution.


## Optional dependencies

The project environment should already contain these packages. Run the command
below only when a package is missing.

```python
# %pip install numpy pandas pyarrow scikit-learn scipy matplotlib
```


In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable, Sequence
import hashlib
import json
import math
import platform
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

LOCAL_TIMEZONE = "Europe/Copenhagen"
TARGET_COLUMN = "actual_wind_mwh"
KEY_COLUMNS = (
    "panel",
    "fold_id",
    "timestamp_utc",
    "price_area",
)

NOTEBOOK_02_OUTPUT_RUN = "notebook_02_v5_full"
NOTEBOOK_03_OUTPUT_RUN = "notebook_03_probabilistic"

QUANTILE_LEVELS = (
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
)

CENTRAL_INTERVALS = (
    0.50,
    0.80,
    0.90,
)

PROVIDER_MAP = {
    "d2": (
        "gfs",
        "icon_eu",
        "ecmwf",
        "gem_global",
        "jma_gsm",
    ),
    "exact": (
        "gfs",
        "icon_eu",
        "ecmwf",
        "dmi",
        "arpege_europe",
        "ukmo_global",
        "cma_grapes",
    ),
}

# Deterministic centres retained from Notebook 02.
CANDIDATE_CENTRES = {
    ("d2", "DK1"): (
        "ensemble::linear_stack",
        "ensemble::hedge",
        "pooled_lightgbm",
    ),
    ("d2", "DK2"): (
        "pooled_lightgbm",
        "ensemble::equal_weight",
        "ensemble::hedge",
    ),
    ("exact", "DK1"): (
        "ensemble::hedge",
        "ensemble::dirichlet_mae",
        "pooled_lightgbm",
    ),
    ("exact", "DK2"): (
        "ensemble::hedge",
        "ensemble::inverse_mae",
        "pooled_lightgbm",
        "ensemble::equal_weight",
    ),
}

PRIMARY_CENTRES = {
    ("d2", "DK1"): "ensemble::linear_stack",
    ("d2", "DK2"): "pooled_lightgbm",
    ("exact", "DK1"): "ensemble::hedge",
    ("exact", "DK2"): "ensemble::hedge",
}

MIN_HISTORY_ROWS = 500
CONDITIONAL_MIN_COMBINATION_ROWS = 20
CONDITIONAL_MIN_HOUR_ROWS = 24
CONDITIONAL_MIN_REGIME_ROWS = 100

GBDT_MAX_ITER = 250
GBDT_LEARNING_RATE = 0.05
GBDT_MAX_LEAF_NODES = 15
GBDT_MIN_SAMPLES_LEAF = 30
GBDT_L2_REGULARIZATION = 1.0

CONFORMAL_TRAIN_FRACTION = 0.70
MIN_CONFORMAL_TRAIN_ROWS = 400
MIN_CONFORMAL_CALIBRATION_ROWS = 150

SCENARIO_COUNT = 250
SCENARIO_SCALE_LOWER = 0.60
SCENARIO_SCALE_UPPER = 1.80
ENERGY_SCORE_PAIR_DRAWS = 500

RUN_QUANTILE_GBDT = True
RUN_CONFORMAL_GBDT = True
RUN_SCENARIOS = True

print(
    {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "random_seed": RANDOM_SEED,
        "notebook_02_output_run": NOTEBOOK_02_OUTPUT_RUN,
        "notebook_03_output_run": NOTEBOOK_03_OUTPUT_RUN,
        "quantiles": QUANTILE_LEVELS,
        "scenario_count": SCENARIO_COUNT,
    }
)


In [ ]:
# ============================================================
# 2. PROJECT ROOT AND OUTPUT DIRECTORIES
# ============================================================

def find_project_root(
    starting_path: Path | None = None,
) -> Path:
    current = Path(starting_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            candidate.joinpath("pyproject.toml").exists()
            and candidate.joinpath("data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected pyproject.toml and data/."
    )


PROJECT_ROOT = find_project_root()

NOTEBOOK_02_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_02_OUTPUT_RUN
)

NOTEBOOK_02_PREDICTION_DIR = (
    NOTEBOOK_02_DIR
    / "predictions"
)

NOTEBOOK_02_AUDIT_DIR = (
    NOTEBOOK_02_DIR
    / "audits"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_03_OUTPUT_RUN
)

AUDIT_DIR = OUTPUT_DIR / "audits"
FIGURE_DIR = OUTPUT_DIR / "figures"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
SCENARIO_DIR = OUTPUT_DIR / "scenarios"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in (
    AUDIT_DIR,
    FIGURE_DIR,
    PREDICTION_DIR,
    SCENARIO_DIR,
    TABLE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:", PROJECT_ROOT)
print("Notebook 02 inputs:", NOTEBOOK_02_DIR)
print("Notebook 03 outputs:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 3. LOAD AND HASH NOTEBOOK 02 OUTPUTS
# ============================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


prediction_path = (
    NOTEBOOK_02_PREDICTION_DIR
    / "notebook_02_test_predictions.parquet"
)

verification_path = (
    NOTEBOOK_02_AUDIT_DIR
    / "notebook_02_verification.json"
)

if not prediction_path.exists():
    raise FileNotFoundError(
        f"Notebook 02 prediction file not found: {prediction_path}"
    )

if not verification_path.exists():
    raise FileNotFoundError(
        f"Notebook 02 verification file not found: {verification_path}"
    )

with verification_path.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_02_verification = json.load(file)

if notebook_02_verification.get(
    "verification_status"
) != "MODELLING_COMPLETE":
    raise RuntimeError(
        "Notebook 02 is not verified as MODELLING_COMPLETE."
    )

all_test_predictions = pd.read_parquet(
    prediction_path
)

all_test_predictions["timestamp_utc"] = pd.to_datetime(
    all_test_predictions["timestamp_utc"],
    utc=True,
)

if "forecast_origin_utc" in all_test_predictions:
    all_test_predictions["forecast_origin_utc"] = pd.to_datetime(
        all_test_predictions["forecast_origin_utc"],
        utc=True,
        errors="coerce",
    )

input_manifest = {
    "prediction_path": str(prediction_path),
    "prediction_sha256": sha256_file(prediction_path),
    "prediction_shape": list(all_test_predictions.shape),
    "notebook_02_verification_path": str(verification_path),
    "notebook_02_verification_status": (
        notebook_02_verification.get(
            "verification_status"
        )
    ),
    "notebook_02_d2_folds": notebook_02_verification.get(
        "d2_folds"
    ),
    "notebook_02_exact_folds": notebook_02_verification.get(
        "exact_folds"
    ),
}

print(json.dumps(input_manifest, indent=2))


In [ ]:
# ============================================================
# 4. INPUT VALIDATION
# ============================================================

required_columns = {
    "panel",
    "fold_id",
    "timestamp_utc",
    "price_area",
    TARGET_COLUMN,
    "model",
    "prediction",
}

missing_columns = (
    required_columns
    - set(all_test_predictions.columns)
)

if missing_columns:
    raise RuntimeError(
        f"Missing required prediction columns: "
        f"{sorted(missing_columns)}"
    )

all_test_predictions[TARGET_COLUMN] = pd.to_numeric(
    all_test_predictions[TARGET_COLUMN],
    errors="coerce",
)

all_test_predictions["prediction"] = pd.to_numeric(
    all_test_predictions["prediction"],
    errors="coerce",
)

all_test_predictions["fold_id"] = pd.to_numeric(
    all_test_predictions["fold_id"],
    errors="raise",
).astype(int)

duplicate_mask = all_test_predictions.duplicated(
    [
        "panel",
        "fold_id",
        "timestamp_utc",
        "price_area",
        "model",
    ],
    keep=False,
)

if duplicate_mask.any():
    duplicate_examples = (
        all_test_predictions.loc[
            duplicate_mask,
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
                "model",
            ],
        ]
        .head(20)
        .to_dict("records")
    )

    raise RuntimeError(
        "Duplicate model predictions were found. "
        f"Examples: {duplicate_examples}"
    )

# The observed target must always be present.
missing_target_rows = int(
    all_test_predictions[TARGET_COLUMN].isna().sum()
)

if missing_target_rows:
    raise RuntimeError(
        "Missing actual wind values were found in "
        f"{missing_target_rows} rows."
    )

# Some Notebook 02 baselines are intentionally sparse. Examples include
# persistence forecasts whose required lag is unavailable and the Energinet
# benchmark on timestamps where no benchmark value exists. Audit such rows
# rather than blocking the probabilistic pipeline.
missing_prediction_audit = (
    all_test_predictions.assign(
        prediction_missing=(
            all_test_predictions["prediction"].isna()
        )
    )
    .groupby(
        [
            "panel",
            "fold_id",
            "price_area",
            "model",
        ],
        as_index=False,
    )
    .agg(
        rows=("timestamp_utc", "size"),
        missing_predictions=(
            "prediction_missing",
            "sum",
        ),
    )
)

missing_prediction_audit["missing_fraction"] = (
    missing_prediction_audit["missing_predictions"]
    / missing_prediction_audit["rows"]
)

missing_prediction_audit.to_csv(
    AUDIT_DIR
    / "notebook_03_input_missing_prediction_audit.csv",
    index=False,
)

dropped_missing_prediction_rows = int(
    all_test_predictions["prediction"].isna().sum()
)

print(
    {
        "input_rows": int(len(all_test_predictions)),
        "missing_prediction_rows": (
            dropped_missing_prediction_rows
        ),
    }
)

if dropped_missing_prediction_rows:
    display(
        missing_prediction_audit.loc[
            missing_prediction_audit[
                "missing_predictions"
            ].gt(0)
        ].sort_values(
            [
                "panel",
                "price_area",
                "model",
                "fold_id",
            ]
        )
    )

# Every downstream probabilistic calculation requires a numeric deterministic
# prediction, so remove only the sparse unusable rows.
all_test_predictions = (
    all_test_predictions.loc[
        all_test_predictions["prediction"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

fold_summary = (
    all_test_predictions.groupby(
        ["panel", "fold_id"],
        as_index=False,
    )
    .agg(
        first_timestamp_utc=(
            "timestamp_utc",
            "min",
        ),
        last_timestamp_utc=(
            "timestamp_utc",
            "max",
        ),
        price_areas=(
            "price_area",
            "nunique",
        ),
        models=(
            "model",
            "nunique",
        ),
        usable_rows=(
            "timestamp_utc",
            "size",
        ),
    )
)

display(fold_summary)


In [ ]:
# ============================================================
# 5. CHECK DETERMINISTIC-CENTRE AVAILABILITY AND COMPLETENESS
# ============================================================

available_models = (
    all_test_predictions.groupby(
        ["panel", "price_area"]
    )["model"]
    .apply(set)
    .to_dict()
)

centre_rows = []

for key, models in CANDIDATE_CENTRES.items():
    panel_name, price_area = key
    available = available_models.get(key, set())

    for model_name in models:
        source_audit = missing_prediction_audit.loc[
            missing_prediction_audit["panel"].eq(
                panel_name
            )
            & missing_prediction_audit["price_area"].eq(
                price_area
            )
            & missing_prediction_audit["model"].eq(
                model_name
            )
        ]

        missing_predictions = int(
            source_audit["missing_predictions"].sum()
        )

        centre_rows.append(
            {
                "panel": panel_name,
                "price_area": price_area,
                "model": model_name,
                "available": model_name in available,
                "missing_predictions": missing_predictions,
                "complete": (
                    model_name in available
                    and missing_predictions == 0
                ),
                "primary": (
                    PRIMARY_CENTRES[key]
                    == model_name
                ),
            }
        )

centre_availability = pd.DataFrame(centre_rows)

centre_availability.to_csv(
    AUDIT_DIR
    / "notebook_03_centre_availability.csv",
    index=False,
)

display(centre_availability)

invalid_centres = centre_availability.loc[
    ~centre_availability["complete"]
]

if not invalid_centres.empty:
    raise RuntimeError(
        "Required deterministic centres are missing or "
        "contain incomplete predictions: "
        f"{invalid_centres.to_dict('records')}"
    )


## Provider-disagreement features

Provider disagreement is calculated from the provider-specific out-of-sample
forecasts at each delivery timestamp. It is not calculated from actual wind and
is therefore available at forecast time.

The probabilistic models use disagreement as a conditional uncertainty signal.


In [ ]:
# ============================================================
# 6. PROVIDER-DISAGREEMENT FEATURES
# ============================================================

def provider_disagreement_features(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    blocks = []

    for panel_name, providers in PROVIDER_MAP.items():
        selected = predictions.loc[
            predictions["panel"].eq(panel_name)
            & predictions["model"].isin(providers),
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
                "model",
                "prediction",
            ],
        ].copy()

        wide = selected.pivot(
            index=[
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
            ],
            columns="model",
            values="prediction",
        )

        values = wide.to_numpy(dtype=float)

        feature_block = wide.reset_index()[
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
            ]
        ].copy()

        feature_block["expert_count"] = np.sum(
            np.isfinite(values),
            axis=1,
        )

        feature_block["expert_mean"] = np.nanmean(
            values,
            axis=1,
        )

        feature_block["expert_median"] = np.nanmedian(
            values,
            axis=1,
        )

        feature_block["expert_std"] = np.nanstd(
            values,
            axis=1,
            ddof=0,
        )

        feature_block["expert_min"] = np.nanmin(
            values,
            axis=1,
        )

        feature_block["expert_max"] = np.nanmax(
            values,
            axis=1,
        )

        feature_block["expert_range"] = (
            feature_block["expert_max"]
            - feature_block["expert_min"]
        )

        q25 = np.nanquantile(
            values,
            0.25,
            axis=1,
        )

        q75 = np.nanquantile(
            values,
            0.75,
            axis=1,
        )

        feature_block["expert_iqr"] = q75 - q25

        denominator = np.maximum(
            np.abs(
                feature_block["expert_mean"]
                .to_numpy(dtype=float)
            ),
            1.0,
        )

        feature_block["expert_cv"] = (
            feature_block["expert_std"]
            .to_numpy(dtype=float)
            / denominator
        )

        blocks.append(feature_block)

    return pd.concat(
        blocks,
        ignore_index=True,
    )


disagreement_features = provider_disagreement_features(
    all_test_predictions
)

disagreement_features.to_parquet(
    PREDICTION_DIR
    / "notebook_03_provider_disagreement_features.parquet",
    index=False,
)

display(
    disagreement_features.groupby(
        ["panel", "price_area"],
        as_index=False,
    ).agg(
        observations=("timestamp_utc", "size"),
        mean_expert_std=("expert_std", "mean"),
        mean_expert_range=("expert_range", "mean"),
        minimum_expert_count=("expert_count", "min"),
    )
)


In [ ]:
# ============================================================
# 7. BUILD THE DETERMINISTIC-CENTRE DATASET
# ============================================================

def selected_centre_names() -> set[str]:
    return {
        model_name
        for models in CANDIDATE_CENTRES.values()
        for model_name in models
    }


centre_predictions = all_test_predictions.loc[
    all_test_predictions["model"].isin(
        selected_centre_names()
    ),
    [
        "panel",
        "fold_id",
        "timestamp_utc",
        "price_area",
        TARGET_COLUMN,
        "model",
        "prediction",
    ],
].copy()

centre_predictions = centre_predictions.merge(
    disagreement_features,
    on=[
        "panel",
        "fold_id",
        "timestamp_utc",
        "price_area",
    ],
    how="left",
    validate="many_to_one",
)

local_timestamp = (
    centre_predictions["timestamp_utc"]
    .dt.tz_convert(LOCAL_TIMEZONE)
)

centre_predictions["delivery_date_local"] = (
    local_timestamp.dt.normalize()
)

centre_predictions["hour_local"] = (
    local_timestamp.dt.hour
)

centre_predictions["weekday_local"] = (
    local_timestamp.dt.weekday
)

centre_predictions["month_local"] = (
    local_timestamp.dt.month
)

centre_predictions["day_of_year_local"] = (
    local_timestamp.dt.dayofyear
)

centre_predictions["hour_sin"] = np.sin(
    2.0
    * np.pi
    * centre_predictions["hour_local"]
    / 24.0
)

centre_predictions["hour_cos"] = np.cos(
    2.0
    * np.pi
    * centre_predictions["hour_local"]
    / 24.0
)

centre_predictions["doy_sin"] = np.sin(
    2.0
    * np.pi
    * centre_predictions["day_of_year_local"]
    / 365.25
)

centre_predictions["doy_cos"] = np.cos(
    2.0
    * np.pi
    * centre_predictions["day_of_year_local"]
    / 365.25
)

centre_predictions["prediction_log1p"] = np.log1p(
    np.maximum(
        centre_predictions["prediction"],
        0.0,
    )
)

centre_predictions["residual"] = (
    centre_predictions[TARGET_COLUMN]
    - centre_predictions["prediction"]
)

feature_columns = [
    "prediction",
    "prediction_log1p",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
    "weekday_local",
    "month_local",
    "expert_mean",
    "expert_median",
    "expert_std",
    "expert_range",
    "expert_iqr",
    "expert_cv",
]

if centre_predictions[
    [
        TARGET_COLUMN,
        "prediction",
        "residual",
    ]
].isna().any().any():
    raise RuntimeError(
        "Missing target, prediction, or residual values "
        "in the centre dataset."
    )

display(
    centre_predictions.groupby(
        ["panel", "price_area", "model", "fold_id"],
        as_index=False,
    ).agg(
        observations=("timestamp_utc", "size"),
        residual_mean=("residual", "mean"),
        residual_mae=("residual", lambda x: float(np.mean(np.abs(x)))),
        mean_disagreement=("expert_std", "mean"),
    )
)


In [ ]:
# ============================================================
# 8. SEQUENTIAL CALIBRATION MANIFEST
# ============================================================

@dataclass(frozen=True)
class CalibrationTask:
    panel: str
    price_area: str
    model: str
    evaluation_fold: int
    history_rows: int
    evaluation_rows: int
    history_last_timestamp_utc: pd.Timestamp
    evaluation_first_timestamp_utc: pd.Timestamp


calibration_tasks: list[CalibrationTask] = []

for (
    panel_name,
    price_area,
), model_names in CANDIDATE_CENTRES.items():
    for model_name in model_names:
        selected = centre_predictions.loc[
            centre_predictions["panel"].eq(panel_name)
            & centre_predictions["price_area"].eq(price_area)
            & centre_predictions["model"].eq(model_name)
        ].copy()

        fold_ids = sorted(
            selected["fold_id"].unique()
        )

        for evaluation_fold in fold_ids[1:]:
            history = selected.loc[
                selected["fold_id"] < evaluation_fold
            ]

            evaluation = selected.loc[
                selected["fold_id"].eq(evaluation_fold)
            ]

            if len(history) < MIN_HISTORY_ROWS:
                raise RuntimeError(
                    f"Insufficient calibration history for "
                    f"{panel_name}/{price_area}/{model_name}/"
                    f"fold {evaluation_fold}: {len(history)} rows."
                )

            history_last = history["timestamp_utc"].max()
            evaluation_first = evaluation["timestamp_utc"].min()

            if history_last >= evaluation_first:
                raise RuntimeError(
                    "Calibration history overlaps or follows the "
                    "evaluation fold for "
                    f"{panel_name}/{price_area}/{model_name}/"
                    f"fold {evaluation_fold}."
                )

            calibration_tasks.append(
                CalibrationTask(
                    panel=panel_name,
                    price_area=price_area,
                    model=model_name,
                    evaluation_fold=int(evaluation_fold),
                    history_rows=int(len(history)),
                    evaluation_rows=int(len(evaluation)),
                    history_last_timestamp_utc=history_last,
                    evaluation_first_timestamp_utc=evaluation_first,
                )
            )


calibration_manifest = pd.DataFrame(
    [
        task.__dict__
        for task in calibration_tasks
    ]
)

calibration_manifest.to_csv(
    AUDIT_DIR
    / "notebook_03_calibration_manifest.csv",
    index=False,
)

display(calibration_manifest)


## Probabilistic methods

The notebook evaluates three methods for every candidate deterministic centre:

1. **Global empirical residual quantiles** — unconditional residual quantiles
   from prior folds.
2. **Conditional empirical residual quantiles** — hierarchical quantiles by
   local hour and provider-disagreement regime, with explicit fallback levels.
3. **Quantile GBDT** — nonlinear residual quantile regressions using the point
   forecast, calendar variables, and provider disagreement.

For the four primary deterministic centres, the notebook additionally computes
**conformalized quantile GBDT** intervals using a chronological train/calibration
split within prior folds.


In [ ]:
# ============================================================
# 9. PROBABILISTIC METRIC UTILITIES
# ============================================================

def pinball_loss(
    actual: np.ndarray,
    forecast: np.ndarray,
    quantile: float,
) -> np.ndarray:
    error = actual - forecast

    return np.maximum(
        quantile * error,
        (quantile - 1.0) * error,
    )


def finite_sample_conformal_quantile(
    scores: np.ndarray,
    alpha: float,
) -> float:
    values = np.asarray(
        scores,
        dtype=float,
    )
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return 0.0

    level = math.ceil(
        (len(values) + 1)
        * (1.0 - alpha)
    ) / len(values)

    level = min(
        max(level, 0.0),
        1.0,
    )

    try:
        return float(
            np.quantile(
                values,
                level,
                method="higher",
            )
        )
    except TypeError:
        return float(
            np.quantile(
                values,
                level,
                interpolation="higher",
            )
        )


def enforce_non_crossing(
    matrix: np.ndarray,
) -> np.ndarray:
    values = np.asarray(
        matrix,
        dtype=float,
    )

    return np.sort(
        values,
        axis=1,
    )


def stable_seed(
    *parts: Any,
) -> int:
    payload = "|".join(
        str(part)
        for part in parts
    ).encode("utf-8")

    digest = hashlib.sha256(
        payload
    ).hexdigest()

    return int(digest[:8], 16)


def quantile_column(
    quantile: float,
) -> str:
    return (
        "q"
        + f"{quantile:.2f}".replace(
            ".",
            "_",
        )
    )


In [ ]:
# ============================================================
# 10. EMPIRICAL RESIDUAL QUANTILE METHODS
# ============================================================

def global_empirical_quantiles(
    history: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> np.ndarray:
    residual_quantiles = (
        history["residual"]
        .quantile(QUANTILE_LEVELS)
        .to_numpy(dtype=float)
    )

    matrix = (
        evaluation["prediction"]
        .to_numpy(dtype=float)[:, None]
        + residual_quantiles[None, :]
    )

    matrix = np.maximum(
        matrix,
        0.0,
    )

    return enforce_non_crossing(matrix)


def disagreement_regime_thresholds(
    history: pd.DataFrame,
) -> tuple[float, float]:
    values = history["expert_std"].dropna()

    if values.empty:
        return (0.0, 0.0)

    lower, upper = values.quantile(
        [1.0 / 3.0, 2.0 / 3.0]
    )

    return float(lower), float(upper)


def assign_disagreement_regime(
    frame: pd.DataFrame,
    thresholds: tuple[float, float],
) -> pd.Series:
    lower, upper = thresholds

    return pd.cut(
        frame["expert_std"],
        bins=[
            -np.inf,
            lower,
            upper,
            np.inf,
        ],
        labels=[
            "low",
            "medium",
            "high",
        ],
        include_lowest=True,
    ).astype("string")


def conditional_empirical_quantiles(
    history: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> np.ndarray:
    history = history.copy()
    evaluation = evaluation.copy()

    thresholds = disagreement_regime_thresholds(
        history
    )

    history["disagreement_regime"] = (
        assign_disagreement_regime(
            history,
            thresholds,
        )
    )

    evaluation["disagreement_regime"] = (
        assign_disagreement_regime(
            evaluation,
            thresholds,
        )
    )

    global_values = (
        history["residual"]
        .quantile(QUANTILE_LEVELS)
        .to_numpy(dtype=float)
    )

    combination_counts = (
        history.groupby(
            [
                "hour_local",
                "disagreement_regime",
            ],
            observed=True,
        )
        .size()
    )

    hour_counts = (
        history.groupby(
            "hour_local"
        )
        .size()
    )

    regime_counts = (
        history.groupby(
            "disagreement_regime",
            observed=True,
        )
        .size()
    )

    combination_quantiles = {
        key: group["residual"]
        .quantile(QUANTILE_LEVELS)
        .to_numpy(dtype=float)
        for key, group in history.groupby(
            [
                "hour_local",
                "disagreement_regime",
            ],
            observed=True,
        )
    }

    hour_quantiles = {
        key: group["residual"]
        .quantile(QUANTILE_LEVELS)
        .to_numpy(dtype=float)
        for key, group in history.groupby(
            "hour_local"
        )
    }

    regime_quantiles = {
        key: group["residual"]
        .quantile(QUANTILE_LEVELS)
        .to_numpy(dtype=float)
        for key, group in history.groupby(
            "disagreement_regime",
            observed=True,
        )
    }

    selected_quantiles = []

    for row in evaluation[
        [
            "hour_local",
            "disagreement_regime",
        ]
    ].itertuples(index=False):
        combination_key = (
            int(row.hour_local),
            str(row.disagreement_regime),
        )

        hour_key = int(
            row.hour_local
        )

        regime_key = str(
            row.disagreement_regime
        )

        if (
            combination_counts.get(
                combination_key,
                0,
            )
            >= CONDITIONAL_MIN_COMBINATION_ROWS
        ):
            residual_values = (
                combination_quantiles[
                    combination_key
                ]
            )
        elif (
            hour_counts.get(
                hour_key,
                0,
            )
            >= CONDITIONAL_MIN_HOUR_ROWS
        ):
            residual_values = (
                hour_quantiles[
                    hour_key
                ]
            )
        elif (
            regime_counts.get(
                regime_key,
                0,
            )
            >= CONDITIONAL_MIN_REGIME_ROWS
        ):
            residual_values = (
                regime_quantiles[
                    regime_key
                ]
            )
        else:
            residual_values = global_values

        selected_quantiles.append(
            residual_values
        )

    residual_matrix = np.vstack(
        selected_quantiles
    )

    matrix = (
        evaluation["prediction"]
        .to_numpy(dtype=float)[:, None]
        + residual_matrix
    )

    matrix = np.maximum(
        matrix,
        0.0,
    )

    return enforce_non_crossing(matrix)


In [ ]:
# ============================================================
# 11. QUANTILE GBDT UTILITIES
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

try:
    from sklearn.ensemble import (
        HistGradientBoostingRegressor,
    )

    QUANTILE_REGRESSOR_BACKEND = (
        "HistGradientBoostingRegressor"
    )
except ImportError:
    HistGradientBoostingRegressor = None

from sklearn.ensemble import (
    GradientBoostingRegressor,
)


def make_quantile_regressor(
    quantile: float,
    *,
    seed: int,
):
    if HistGradientBoostingRegressor is not None:
        try:
            estimator = HistGradientBoostingRegressor(
                loss="quantile",
                quantile=quantile,
                max_iter=GBDT_MAX_ITER,
                learning_rate=GBDT_LEARNING_RATE,
                max_leaf_nodes=GBDT_MAX_LEAF_NODES,
                min_samples_leaf=GBDT_MIN_SAMPLES_LEAF,
                l2_regularization=GBDT_L2_REGULARIZATION,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
                random_state=seed,
            )

            return make_pipeline(
                SimpleImputer(
                    strategy="median"
                ),
                estimator,
            )
        except TypeError:
            pass

    estimator = GradientBoostingRegressor(
        loss="quantile",
        alpha=quantile,
        n_estimators=GBDT_MAX_ITER,
        learning_rate=GBDT_LEARNING_RATE,
        max_depth=3,
        min_samples_leaf=GBDT_MIN_SAMPLES_LEAF,
        random_state=seed,
    )

    return make_pipeline(
        SimpleImputer(
            strategy="median"
        ),
        estimator,
    )


def fit_predict_quantile_gbdt(
    history: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    seed_prefix: Sequence[Any],
) -> tuple[np.ndarray, dict[float, Any]]:
    training_features = history[
        feature_columns
    ]

    evaluation_features = evaluation[
        feature_columns
    ]

    target_residual = (
        history["residual"]
        .to_numpy(dtype=float)
    )

    residual_predictions = []
    models: dict[float, Any] = {}

    for quantile in QUANTILE_LEVELS:
        model = make_quantile_regressor(
            quantile,
            seed=stable_seed(
                RANDOM_SEED,
                *seed_prefix,
                quantile,
            ),
        )

        model.fit(
            training_features,
            target_residual,
        )

        residual_prediction = model.predict(
            evaluation_features
        )

        residual_predictions.append(
            np.asarray(
                residual_prediction,
                dtype=float,
            )
        )

        models[quantile] = model

    residual_matrix = np.column_stack(
        residual_predictions
    )

    forecast_matrix = (
        evaluation["prediction"]
        .to_numpy(dtype=float)[:, None]
        + residual_matrix
    )

    forecast_matrix = np.maximum(
        forecast_matrix,
        0.0,
    )

    return (
        enforce_non_crossing(
            forecast_matrix
        ),
        models,
    )


In [ ]:
# ============================================================
# 12. CONFORMALIZED QUANTILE GBDT
# ============================================================

SYMMETRIC_QUANTILE_PAIRS = (
    (0.05, 0.95),
    (0.10, 0.90),
    (0.25, 0.75),
)


def chronological_conformal_split(
    history: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = history.sort_values(
        "timestamp_utc"
    ).reset_index(drop=True)

    split_position = int(
        len(ordered)
        * CONFORMAL_TRAIN_FRACTION
    )

    split_position = max(
        split_position,
        MIN_CONFORMAL_TRAIN_ROWS,
    )

    split_position = min(
        split_position,
        len(ordered)
        - MIN_CONFORMAL_CALIBRATION_ROWS,
    )

    if (
        split_position < MIN_CONFORMAL_TRAIN_ROWS
        or (
            len(ordered)
            - split_position
        ) < MIN_CONFORMAL_CALIBRATION_ROWS
    ):
        raise RuntimeError(
            "Insufficient rows for chronological "
            "conformal train/calibration split."
        )

    return (
        ordered.iloc[:split_position].copy(),
        ordered.iloc[split_position:].copy(),
    )


def conformalized_quantile_gbdt(
    history: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    seed_prefix: Sequence[Any],
) -> tuple[np.ndarray, pd.DataFrame]:
    fit_history, calibration = (
        chronological_conformal_split(
            history
        )
    )

    calibration_matrix, models = (
        fit_predict_quantile_gbdt(
            fit_history,
            calibration,
            seed_prefix=(
                *seed_prefix,
                "conformal_fit",
            ),
        )
    )

    evaluation_matrix = (
        np.column_stack(
            [
                models[quantile].predict(
                    evaluation[feature_columns]
                )
                for quantile in QUANTILE_LEVELS
            ]
        )
        + evaluation["prediction"]
        .to_numpy(dtype=float)[:, None]
    )

    evaluation_matrix = np.maximum(
        evaluation_matrix,
        0.0,
    )

    quantile_to_index = {
        quantile: index
        for index, quantile in enumerate(
            QUANTILE_LEVELS
        )
    }

    actual_calibration = calibration[
        TARGET_COLUMN
    ].to_numpy(dtype=float)

    adjustment_rows = []

    for lower_quantile, upper_quantile in (
        SYMMETRIC_QUANTILE_PAIRS
    ):
        lower_index = quantile_to_index[
            lower_quantile
        ]

        upper_index = quantile_to_index[
            upper_quantile
        ]

        alpha = (
            lower_quantile
            + (1.0 - upper_quantile)
        )

        scores = np.maximum(
            calibration_matrix[
                :,
                lower_index,
            ]
            - actual_calibration,
            actual_calibration
            - calibration_matrix[
                :,
                upper_index,
            ],
        )

        adjustment = (
            finite_sample_conformal_quantile(
                scores,
                alpha=alpha,
            )
        )

        evaluation_matrix[
            :,
            lower_index,
        ] -= adjustment

        evaluation_matrix[
            :,
            upper_index,
        ] += adjustment

        adjustment_rows.append(
            {
                "lower_quantile": lower_quantile,
                "upper_quantile": upper_quantile,
                "nominal_coverage": (
                    upper_quantile
                    - lower_quantile
                ),
                "conformal_alpha": alpha,
                "calibration_rows": len(calibration),
                "adjustment": adjustment,
            }
        )

    evaluation_matrix = np.maximum(
        evaluation_matrix,
        0.0,
    )

    return (
        enforce_non_crossing(
            evaluation_matrix
        ),
        pd.DataFrame(adjustment_rows),
    )


In [ ]:
# ============================================================
# 13. WALK-FORWARD PROBABILISTIC FORECAST GENERATION
# ============================================================

def matrix_to_long_predictions(
    evaluation: pd.DataFrame,
    matrix: np.ndarray,
    *,
    probabilistic_method: str,
) -> pd.DataFrame:
    blocks = []

    base_columns = [
        "panel",
        "fold_id",
        "timestamp_utc",
        "delivery_date_local",
        "price_area",
        TARGET_COLUMN,
        "model",
        "prediction",
        "expert_std",
        "expert_range",
        "expert_iqr",
    ]

    for quantile_index, quantile in enumerate(
        QUANTILE_LEVELS
    ):
        block = evaluation[
            base_columns
        ].copy()

        block = block.rename(
            columns={
                "model": "centre_model",
                "prediction": (
                    "centre_prediction"
                ),
            }
        )

        block["probabilistic_method"] = (
            probabilistic_method
        )

        block["quantile"] = quantile
        block["probabilistic_prediction"] = (
            matrix[:, quantile_index]
        )

        blocks.append(block)

    return pd.concat(
        blocks,
        ignore_index=True,
    )


probabilistic_blocks = []
conformal_adjustment_blocks = []

for task in calibration_tasks:
    task_mask = (
        centre_predictions["panel"].eq(
            task.panel
        )
        & centre_predictions[
            "price_area"
        ].eq(task.price_area)
        & centre_predictions[
            "model"
        ].eq(task.model)
    )

    history = centre_predictions.loc[
        task_mask
        & (
            centre_predictions["fold_id"]
            < task.evaluation_fold
        )
    ].copy()

    evaluation = centre_predictions.loc[
        task_mask
        & centre_predictions["fold_id"].eq(
            task.evaluation_fold
        )
    ].copy()

    global_matrix = global_empirical_quantiles(
        history,
        evaluation,
    )

    probabilistic_blocks.append(
        matrix_to_long_predictions(
            evaluation,
            global_matrix,
            probabilistic_method=(
                "empirical_global"
            ),
        )
    )

    conditional_matrix = (
        conditional_empirical_quantiles(
            history,
            evaluation,
        )
    )

    probabilistic_blocks.append(
        matrix_to_long_predictions(
            evaluation,
            conditional_matrix,
            probabilistic_method=(
                "empirical_conditional"
            ),
        )
    )

    if RUN_QUANTILE_GBDT:
        gbdt_matrix, _ = (
            fit_predict_quantile_gbdt(
                history,
                evaluation,
                seed_prefix=(
                    task.panel,
                    task.price_area,
                    task.model,
                    task.evaluation_fold,
                    "all_history",
                ),
            )
        )

        probabilistic_blocks.append(
            matrix_to_long_predictions(
                evaluation,
                gbdt_matrix,
                probabilistic_method=(
                    "quantile_gbdt"
                ),
            )
        )

    is_primary = (
        PRIMARY_CENTRES[
            (
                task.panel,
                task.price_area,
            )
        ]
        == task.model
    )

    if (
        RUN_CONFORMAL_GBDT
        and is_primary
    ):
        cqr_matrix, adjustments = (
            conformalized_quantile_gbdt(
                history,
                evaluation,
                seed_prefix=(
                    task.panel,
                    task.price_area,
                    task.model,
                    task.evaluation_fold,
                ),
            )
        )

        probabilistic_blocks.append(
            matrix_to_long_predictions(
                evaluation,
                cqr_matrix,
                probabilistic_method=(
                    "conformal_quantile_gbdt"
                ),
            )
        )

        adjustments["panel"] = task.panel
        adjustments["price_area"] = (
            task.price_area
        )
        adjustments["centre_model"] = (
            task.model
        )
        adjustments["fold_id"] = (
            task.evaluation_fold
        )

        conformal_adjustment_blocks.append(
            adjustments
        )


probabilistic_predictions = pd.concat(
    probabilistic_blocks,
    ignore_index=True,
)

conformal_adjustments = (
    pd.concat(
        conformal_adjustment_blocks,
        ignore_index=True,
    )
    if conformal_adjustment_blocks
    else pd.DataFrame()
)

probabilistic_predictions.to_parquet(
    PREDICTION_DIR
    / "notebook_03_quantile_predictions.parquet",
    index=False,
)

conformal_adjustments.to_csv(
    TABLE_DIR
    / "notebook_03_conformal_adjustments.csv",
    index=False,
)

print(
    "Probabilistic prediction rows:",
    len(probabilistic_predictions),
)

display(
    probabilistic_predictions.groupby(
        [
            "panel",
            "price_area",
            "centre_model",
            "probabilistic_method",
            "fold_id",
        ],
        as_index=False,
    ).agg(
        timestamps=(
            "timestamp_utc",
            "nunique",
        ),
        quantiles=(
            "quantile",
            "nunique",
        ),
    )
)


In [ ]:
# ============================================================
# 14. QUANTILE, INTERVAL, WIS, AND CRPS EVALUATION
# ============================================================

def approximate_crps_from_quantiles(
    actual: float,
    quantile_values: np.ndarray,
) -> float:
    quantile_values = np.asarray(
        quantile_values,
        dtype=float,
    )

    losses = np.array(
        [
            pinball_loss(
                np.array([actual]),
                np.array([forecast]),
                quantile,
            )[0]
            for quantile, forecast in zip(
                QUANTILE_LEVELS,
                quantile_values,
            )
        ],
        dtype=float,
    )

    if hasattr(np, "trapezoid"):
        integral = np.trapezoid(
            losses,
            x=np.asarray(
                QUANTILE_LEVELS,
                dtype=float,
            ),
        )
    else:
        integral = np.trapz(
            losses,
            x=np.asarray(
                QUANTILE_LEVELS,
                dtype=float,
            ),
        )

    return float(2.0 * integral)


def interval_score(
    actual: float,
    lower: float,
    upper: float,
    alpha: float,
) -> float:
    score = upper - lower

    if actual < lower:
        score += (
            2.0
            / alpha
            * (lower - actual)
        )
    elif actual > upper:
        score += (
            2.0
            / alpha
            * (actual - upper)
        )

    return float(score)


def weighted_interval_score(
    actual: float,
    quantile_lookup: dict[float, float],
) -> float:
    median = quantile_lookup[0.50]

    weighted_sum = (
        0.5
        * abs(actual - median)
    )

    for coverage in CENTRAL_INTERVALS:
        alpha = 1.0 - coverage
        lower_quantile = alpha / 2.0
        upper_quantile = 1.0 - alpha / 2.0

        lower = quantile_lookup[
            round(lower_quantile, 2)
        ]

        upper = quantile_lookup[
            round(upper_quantile, 2)
        ]

        weighted_sum += (
            alpha
            / 2.0
            * interval_score(
                actual,
                lower,
                upper,
                alpha,
            )
        )

    return float(
        weighted_sum
        / (
            len(CENTRAL_INTERVALS)
            + 0.5
        )
    )


quantile_metric_rows = []

for keys, group in probabilistic_predictions.groupby(
    [
        "panel",
        "fold_id",
        "price_area",
        "centre_model",
        "probabilistic_method",
        "quantile",
    ]
):
    (
        panel_name,
        fold_id,
        price_area,
        centre_model,
        method,
        quantile,
    ) = keys

    actual = group[
        TARGET_COLUMN
    ].to_numpy(dtype=float)

    forecast = group[
        "probabilistic_prediction"
    ].to_numpy(dtype=float)

    losses = pinball_loss(
        actual,
        forecast,
        float(quantile),
    )

    quantile_metric_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "centre_model": centre_model,
            "probabilistic_method": method,
            "quantile": float(quantile),
            "pinball_loss": float(
                np.mean(losses)
            ),
            "empirical_cdf": float(
                np.mean(actual <= forecast)
            ),
            "calibration_error": float(
                np.mean(actual <= forecast)
                - float(quantile)
            ),
            "observations": int(len(group)),
        }
    )


quantile_metrics = pd.DataFrame(
    quantile_metric_rows
)

wide_index = [
    "panel",
    "fold_id",
    "timestamp_utc",
    "delivery_date_local",
    "price_area",
    TARGET_COLUMN,
    "centre_model",
    "centre_prediction",
    "probabilistic_method",
]

quantile_wide = (
    probabilistic_predictions.pivot(
        index=wide_index,
        columns="quantile",
        values="probabilistic_prediction",
    )
    .reset_index()
)

quantile_wide.columns.name = None

rename_quantiles = {
    quantile: quantile_column(quantile)
    for quantile in QUANTILE_LEVELS
}

quantile_wide = quantile_wide.rename(
    columns=rename_quantiles
)

row_metric_rows = []

for row in quantile_wide.itertuples(
    index=False
):
    lookup = {
        quantile: float(
            getattr(
                row,
                quantile_column(
                    quantile
                ),
            )
        )
        for quantile in QUANTILE_LEVELS
    }

    actual = float(
        getattr(
            row,
            TARGET_COLUMN,
        )
    )

    metrics = {
        "panel": row.panel,
        "fold_id": row.fold_id,
        "timestamp_utc": row.timestamp_utc,
        "delivery_date_local": row.delivery_date_local,
        "price_area": row.price_area,
        TARGET_COLUMN: actual,
        "centre_model": row.centre_model,
        "centre_prediction": row.centre_prediction,
        "probabilistic_method": row.probabilistic_method,
        "median_absolute_error": abs(
            lookup[0.50] - actual
        ),
        "median_error": (
            lookup[0.50] - actual
        ),
        "wis": weighted_interval_score(
            actual,
            lookup,
        ),
        "crps_approx": (
            approximate_crps_from_quantiles(
                actual,
                np.array(
                    [
                        lookup[q]
                        for q in QUANTILE_LEVELS
                    ]
                ),
            )
        ),
    }

    for coverage in CENTRAL_INTERVALS:
        alpha = 1.0 - coverage
        lower_quantile = round(
            alpha / 2.0,
            2,
        )
        upper_quantile = round(
            1.0 - alpha / 2.0,
            2,
        )

        lower = lookup[lower_quantile]
        upper = lookup[upper_quantile]

        coverage_label = int(
            round(coverage * 100)
        )

        metrics[
            f"coverage_{coverage_label}"
        ] = float(
            lower <= actual <= upper
        )

        metrics[
            f"width_{coverage_label}"
        ] = upper - lower

    row_metric_rows.append(metrics)


probabilistic_row_metrics = pd.DataFrame(
    row_metric_rows
)

probabilistic_fold_metrics = (
    probabilistic_row_metrics.groupby(
        [
            "panel",
            "fold_id",
            "price_area",
            "centre_model",
            "probabilistic_method",
        ],
        as_index=False,
    )
    .agg(
        mean_wis=("wis", "mean"),
        mean_crps=("crps_approx", "mean"),
        median_mae=(
            "median_absolute_error",
            "mean",
        ),
        median_bias=(
            "median_error",
            "mean",
        ),
        coverage_50=("coverage_50", "mean"),
        width_50=("width_50", "mean"),
        coverage_80=("coverage_80", "mean"),
        width_80=("width_80", "mean"),
        coverage_90=("coverage_90", "mean"),
        width_90=("width_90", "mean"),
        observations=("timestamp_utc", "size"),
    )
)

probabilistic_fold_metrics.to_csv(
    TABLE_DIR
    / "notebook_03_probabilistic_metrics_by_fold.csv",
    index=False,
)

quantile_metrics.to_csv(
    TABLE_DIR
    / "notebook_03_quantile_calibration_by_fold.csv",
    index=False,
)

probabilistic_row_metrics.to_parquet(
    PREDICTION_DIR
    / "notebook_03_probabilistic_row_metrics.parquet",
    index=False,
)

display(
    probabilistic_fold_metrics.sort_values(
        [
            "panel",
            "price_area",
            "fold_id",
            "mean_wis",
        ]
    )
)


In [ ]:
# ============================================================
# 15. AGGREGATE PROBABILISTIC RANKING
# ============================================================

probabilistic_summary = (
    probabilistic_fold_metrics.groupby(
        [
            "panel",
            "price_area",
            "centre_model",
            "probabilistic_method",
        ],
        as_index=False,
    )
    .agg(
        mean_wis=("mean_wis", "mean"),
        std_wis=("mean_wis", "std"),
        mean_crps=("mean_crps", "mean"),
        median_mae=("median_mae", "mean"),
        median_bias=("median_bias", "mean"),
        coverage_50=("coverage_50", "mean"),
        width_50=("width_50", "mean"),
        coverage_80=("coverage_80", "mean"),
        width_80=("width_80", "mean"),
        coverage_90=("coverage_90", "mean"),
        width_90=("width_90", "mean"),
        folds=("fold_id", "nunique"),
        observations=("observations", "sum"),
    )
)

probabilistic_summary["coverage_error_50"] = (
    probabilistic_summary["coverage_50"]
    - 0.50
).abs()

probabilistic_summary["coverage_error_80"] = (
    probabilistic_summary["coverage_80"]
    - 0.80
).abs()

probabilistic_summary["coverage_error_90"] = (
    probabilistic_summary["coverage_90"]
    - 0.90
).abs()

probabilistic_summary["wis_rank"] = (
    probabilistic_summary.groupby(
        ["panel", "price_area"]
    )["mean_wis"]
    .rank(
        method="dense",
        ascending=True,
    )
    .astype(int)
)

probabilistic_summary.to_csv(
    TABLE_DIR
    / "notebook_03_probabilistic_metric_summary.csv",
    index=False,
)

display(
    probabilistic_summary.sort_values(
        [
            "panel",
            "price_area",
            "wis_rank",
            "mean_crps",
        ]
    )
)


In [ ]:
# ============================================================
# 16. RELIABILITY AND PIT DIAGNOSTICS
# ============================================================

quantile_reliability = (
    probabilistic_predictions.assign(
        below_forecast=lambda frame: (
            frame[TARGET_COLUMN]
            <= frame["probabilistic_prediction"]
        ).astype(float)
    )
    .groupby(
        [
            "panel",
            "price_area",
            "centre_model",
            "probabilistic_method",
            "quantile",
        ],
        as_index=False,
    )
    .agg(
        empirical_probability=(
            "below_forecast",
            "mean",
        ),
        observations=(
            "below_forecast",
            "size",
        ),
    )
)

quantile_reliability[
    "calibration_error"
] = (
    quantile_reliability[
        "empirical_probability"
    ]
    - quantile_reliability["quantile"]
)


def pit_from_quantile_grid(
    actual: float,
    quantile_values: np.ndarray,
) -> float:
    forecasts = np.asarray(
        quantile_values,
        dtype=float,
    )

    quantiles = np.asarray(
        QUANTILE_LEVELS,
        dtype=float,
    )

    if actual <= forecasts[0]:
        return float(
            quantiles[0]
            * max(
                actual,
                0.0,
            )
            / max(
                forecasts[0],
                1e-9,
            )
        )

    if actual >= forecasts[-1]:
        return float(
            quantiles[-1]
            + (
                1.0
                - quantiles[-1]
            )
            * min(
                (
                    actual
                    - forecasts[-1]
                )
                / max(
                    abs(forecasts[-1]),
                    1.0,
                ),
                1.0,
            )
        )

    return float(
        np.interp(
            actual,
            forecasts,
            quantiles,
        )
    )


pit_rows = []

for row in quantile_wide.itertuples(
    index=False
):
    quantile_values = np.array(
        [
            getattr(
                row,
                quantile_column(q),
            )
            for q in QUANTILE_LEVELS
        ],
        dtype=float,
    )

    pit_rows.append(
        {
            "panel": row.panel,
            "fold_id": row.fold_id,
            "timestamp_utc": row.timestamp_utc,
            "price_area": row.price_area,
            "centre_model": row.centre_model,
            "probabilistic_method": row.probabilistic_method,
            "pit": pit_from_quantile_grid(
                float(
                    getattr(
                        row,
                        TARGET_COLUMN,
                    )
                ),
                quantile_values,
            ),
        }
    )


pit_values = pd.DataFrame(
    pit_rows
)

pit_summary = (
    pit_values.groupby(
        [
            "panel",
            "price_area",
            "centre_model",
            "probabilistic_method",
        ],
        as_index=False,
    )
    .agg(
        pit_mean=("pit", "mean"),
        pit_variance=("pit", "var"),
        pit_below_0_1=(
            "pit",
            lambda values: float(
                np.mean(values < 0.10)
            ),
        ),
        pit_above_0_9=(
            "pit",
            lambda values: float(
                np.mean(values > 0.90)
            ),
        ),
        observations=("pit", "size"),
    )
)

quantile_reliability.to_csv(
    TABLE_DIR
    / "notebook_03_quantile_reliability.csv",
    index=False,
)

pit_values.to_parquet(
    PREDICTION_DIR
    / "notebook_03_pit_values.parquet",
    index=False,
)

pit_summary.to_csv(
    TABLE_DIR
    / "notebook_03_pit_summary.csv",
    index=False,
)

display(pit_summary)


In [ ]:
# ============================================================
# 17. PROBABILISTIC DIAGNOSTIC FIGURES
# ============================================================

import matplotlib.pyplot as plt


def save_current_figure(
    filename: str,
) -> None:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


for panel_name in sorted(
    probabilistic_summary["panel"].unique()
):
    for price_area in sorted(
        probabilistic_summary.loc[
            probabilistic_summary["panel"].eq(
                panel_name
            ),
            "price_area",
        ].unique()
    ):
        selected = (
            probabilistic_summary.loc[
                probabilistic_summary["panel"].eq(
                    panel_name
                )
                & probabilistic_summary[
                    "price_area"
                ].eq(price_area)
            ]
            .sort_values("mean_wis")
            .head(12)
            .copy()
        )

        labels = (
            selected["centre_model"]
            + "\n"
            + selected[
                "probabilistic_method"
            ]
        )

        plt.figure(figsize=(11, 6))
        plt.barh(
            labels[::-1],
            selected["mean_wis"][::-1],
        )
        plt.xlabel("Mean weighted interval score")
        plt.ylabel("Deterministic centre and probabilistic method")
        plt.title(
            f"{panel_name.upper()} {price_area}: "
            "probabilistic ranking"
        )

        save_current_figure(
            f"notebook_03_wis_ranking_"
            f"{panel_name}_{price_area}.png"
        )


for panel_name, price_area in PRIMARY_CENTRES:
    centre_model = PRIMARY_CENTRES[
        (panel_name, price_area)
    ]

    selected = quantile_reliability.loc[
        quantile_reliability["panel"].eq(
            panel_name
        )
        & quantile_reliability[
            "price_area"
        ].eq(price_area)
        & quantile_reliability[
            "centre_model"
        ].eq(centre_model)
    ]

    plt.figure(figsize=(7, 6))

    for method, group in selected.groupby(
        "probabilistic_method"
    ):
        ordered = group.sort_values(
            "quantile"
        )

        plt.plot(
            ordered["quantile"],
            ordered["empirical_probability"],
            marker="o",
            label=method,
        )

    plt.plot(
        [0.0, 1.0],
        [0.0, 1.0],
        linestyle="--",
        label="ideal",
    )

    plt.xlabel("Nominal probability")
    plt.ylabel("Empirical probability")
    plt.title(
        f"{panel_name.upper()} {price_area}: "
        f"reliability for {centre_model}"
    )
    plt.legend()

    save_current_figure(
        f"notebook_03_reliability_"
        f"{panel_name}_{price_area}.png"
    )


## Joint DK1–DK2 scenario generation

Scenarios are generated only for the four primary deterministic centres.

For each evaluation local day:

1. use only residual days from earlier out-of-sample folds;
2. sample a historical residual day jointly for DK1 and DK2;
3. preserve its hourly temporal structure and cross-area dependence;
4. scale each area's residual path by the ratio of current to sampled provider
   disagreement;
5. add the scaled residual path to the deterministic centre and truncate at
   zero.

This produces coherent hourly paths rather than independent marginal draws.


In [ ]:
# ============================================================
# 18. PRIMARY-CENTRE FRAME FOR SCENARIOS
# ============================================================

primary_blocks = []

for (
    panel_name,
    price_area,
), model_name in PRIMARY_CENTRES.items():
    selected = centre_predictions.loc[
        centre_predictions["panel"].eq(
            panel_name
        )
        & centre_predictions[
            "price_area"
        ].eq(price_area)
        & centre_predictions[
            "model"
        ].eq(model_name)
    ].copy()

    selected["centre_model"] = model_name
    primary_blocks.append(selected)


primary_centre_predictions = pd.concat(
    primary_blocks,
    ignore_index=True,
)

primary_centre_predictions.to_parquet(
    PREDICTION_DIR
    / "notebook_03_primary_centre_predictions.parquet",
    index=False,
)

display(
    primary_centre_predictions.groupby(
        [
            "panel",
            "price_area",
            "centre_model",
            "fold_id",
        ],
        as_index=False,
    ).size()
)


In [ ]:
# ============================================================
# 19. JOINT RESIDUAL-DAY LIBRARY
# ============================================================

def complete_joint_day_blocks(
    frame: pd.DataFrame,
) -> list[dict[str, Any]]:
    blocks = []

    for delivery_date, day in frame.groupby(
        "delivery_date_local"
    ):
        timestamp_counts = (
            day.groupby("timestamp_utc")[
                "price_area"
            ]
            .nunique()
        )

        complete_timestamps = (
            timestamp_counts.loc[
                timestamp_counts.eq(2)
            ]
            .index
        )

        day = day.loc[
            day["timestamp_utc"].isin(
                complete_timestamps
            )
        ].copy()

        residual_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values="residual",
        ).sort_index()

        disagreement_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values="expert_std",
        ).sort_index()

        hour_lookup = (
            day.drop_duplicates(
                "timestamp_utc"
            )
            .set_index("timestamp_utc")[
                "hour_local"
            ]
            .reindex(
                residual_wide.index
            )
        )

        if (
            set(residual_wide.columns)
            != {"DK1", "DK2"}
            or residual_wide.isna().any().any()
        ):
            continue

        blocks.append(
            {
                "delivery_date_local": delivery_date,
                "timestamps": residual_wide.index,
                "local_hours": tuple(
                    hour_lookup.astype(int)
                ),
                "residual_matrix": (
                    residual_wide[
                        ["DK1", "DK2"]
                    ]
                    .to_numpy(dtype=float)
                ),
                "mean_disagreement": (
                    disagreement_wide[
                        ["DK1", "DK2"]
                    ]
                    .mean()
                    .to_numpy(dtype=float)
                ),
            }
        )

    return blocks


def choose_candidate_blocks(
    library: list[dict[str, Any]],
    local_hours: tuple[int, ...],
) -> list[dict[str, Any]]:
    exact = [
        block
        for block in library
        if block["local_hours"]
        == local_hours
    ]

    if exact:
        return exact

    same_length = [
        block
        for block in library
        if len(block["local_hours"])
        == len(local_hours)
    ]

    if same_length:
        return same_length

    raise RuntimeError(
        "No historical residual day block matches "
        f"the evaluation-day length {len(local_hours)}."
    )


In [ ]:
# ============================================================
# 20. GENERATE JOINT SCENARIOS
# ============================================================

def generate_panel_fold_scenarios(
    frame: pd.DataFrame,
    *,
    panel_name: str,
    evaluation_fold: int,
) -> pd.DataFrame:
    history = frame.loc[
        frame["panel"].eq(panel_name)
        & (
            frame["fold_id"]
            < evaluation_fold
        )
    ].copy()

    evaluation = frame.loc[
        frame["panel"].eq(panel_name)
        & frame["fold_id"].eq(
            evaluation_fold
        )
    ].copy()

    if history.empty or evaluation.empty:
        raise RuntimeError(
            f"Missing history or evaluation rows for "
            f"{panel_name} fold {evaluation_fold}."
        )

    library = complete_joint_day_blocks(
        history
    )

    if not library:
        raise RuntimeError(
            f"No complete joint residual-day blocks "
            f"for {panel_name} fold {evaluation_fold}."
        )

    rng = np.random.default_rng(
        stable_seed(
            RANDOM_SEED,
            panel_name,
            evaluation_fold,
            "joint_scenarios",
        )
    )

    scenario_blocks = []

    for delivery_date, day in evaluation.groupby(
        "delivery_date_local"
    ):
        day = day.sort_values(
            [
                "timestamp_utc",
                "price_area",
            ]
        )

        centre_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values="prediction",
        ).sort_index()

        actual_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values=TARGET_COLUMN,
        ).sort_index()

        disagreement_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values="expert_std",
        ).sort_index()

        model_wide = day.pivot(
            index="timestamp_utc",
            columns="price_area",
            values="centre_model",
        ).sort_index()

        hour_lookup = (
            day.drop_duplicates(
                "timestamp_utc"
            )
            .set_index("timestamp_utc")[
                "hour_local"
            ]
            .reindex(
                centre_wide.index
            )
        )

        if (
            set(centre_wide.columns)
            != {"DK1", "DK2"}
            or centre_wide.isna().any().any()
        ):
            raise RuntimeError(
                "Primary-centre evaluation day is "
                f"incomplete: {panel_name}, {delivery_date}."
            )

        local_hours = tuple(
            hour_lookup.astype(int)
        )

        candidates = choose_candidate_blocks(
            library,
            local_hours,
        )

        current_disagreement = (
            disagreement_wide[
                ["DK1", "DK2"]
            ]
            .mean()
            .to_numpy(dtype=float)
        )

        centre_matrix = (
            centre_wide[
                ["DK1", "DK2"]
            ]
            .to_numpy(dtype=float)
        )

        actual_matrix = (
            actual_wide[
                ["DK1", "DK2"]
            ]
            .to_numpy(dtype=float)
        )

        for scenario_id in range(
            1,
            SCENARIO_COUNT + 1,
        ):
            sampled_block = candidates[
                int(
                    rng.integers(
                        0,
                        len(candidates),
                    )
                )
            ]

            sampled_residual = (
                sampled_block[
                    "residual_matrix"
                ].copy()
            )

            sampled_disagreement = np.maximum(
                sampled_block[
                    "mean_disagreement"
                ],
                1e-6,
            )

            scale = np.clip(
                current_disagreement
                / sampled_disagreement,
                SCENARIO_SCALE_LOWER,
                SCENARIO_SCALE_UPPER,
            )

            scenario_matrix = np.maximum(
                centre_matrix
                + sampled_residual
                * scale[None, :],
                0.0,
            )

            for area_index, price_area in enumerate(
                ("DK1", "DK2")
            ):
                block = pd.DataFrame(
                    {
                        "panel": panel_name,
                        "fold_id": evaluation_fold,
                        "delivery_date_local": delivery_date,
                        "timestamp_utc": centre_wide.index,
                        "price_area": price_area,
                        "scenario_id": scenario_id,
                        "centre_model": model_wide[
                            price_area
                        ].to_numpy(),
                        "centre_prediction": centre_matrix[
                            :,
                            area_index,
                        ],
                        TARGET_COLUMN: actual_matrix[
                            :,
                            area_index,
                        ],
                        "scenario_wind_mwh": scenario_matrix[
                            :,
                            area_index,
                        ],
                        "sampled_residual_date_local": sampled_block[
                            "delivery_date_local"
                        ],
                        "disagreement_scale": scale[
                            area_index
                        ],
                    }
                )

                scenario_blocks.append(block)

    return pd.concat(
        scenario_blocks,
        ignore_index=True,
    )


scenario_blocks = []

if RUN_SCENARIOS:
    for panel_name in (
        "d2",
        "exact",
    ):
        panel_folds = sorted(
            primary_centre_predictions.loc[
                primary_centre_predictions[
                    "panel"
                ].eq(panel_name),
                "fold_id",
            ].unique()
        )

        for evaluation_fold in panel_folds[1:]:
            scenario_blocks.append(
                generate_panel_fold_scenarios(
                    primary_centre_predictions,
                    panel_name=panel_name,
                    evaluation_fold=int(
                        evaluation_fold
                    ),
                )
            )


joint_scenarios = (
    pd.concat(
        scenario_blocks,
        ignore_index=True,
    )
    if scenario_blocks
    else pd.DataFrame()
)

if not joint_scenarios.empty:
    joint_scenarios.to_parquet(
        SCENARIO_DIR
        / "notebook_03_joint_wind_scenarios.parquet",
        index=False,
    )

print(
    "Joint scenario rows:",
    len(joint_scenarios),
)


In [ ]:
# ============================================================
# 21. MARGINAL SCENARIO EVALUATION
# ============================================================

def ensemble_crps(
    ensemble: np.ndarray,
    actual: float,
) -> float:
    values = np.sort(
        np.asarray(
            ensemble,
            dtype=float,
        )
    )

    n = len(values)

    if n == 0:
        return np.nan

    first_term = float(
        np.mean(
            np.abs(values - actual)
        )
    )

    ranks = np.arange(
        1,
        n + 1,
        dtype=float,
    )

    pair_term = float(
        np.sum(
            (
                2.0 * ranks
                - n
                - 1.0
            )
            * values
        )
        / (n ** 2)
    )

    return first_term - pair_term


scenario_marginal_rows = []

if not joint_scenarios.empty:
    for keys, group in joint_scenarios.groupby(
        [
            "panel",
            "fold_id",
            "timestamp_utc",
            "price_area",
            "centre_model",
        ]
    ):
        (
            panel_name,
            fold_id,
            timestamp_utc,
            price_area,
            centre_model,
        ) = keys

        values = group[
            "scenario_wind_mwh"
        ].to_numpy(dtype=float)

        actual = float(
            group[TARGET_COLUMN].iloc[0]
        )

        q05, q10, q50, q90, q95 = (
            np.quantile(
                values,
                [
                    0.05,
                    0.10,
                    0.50,
                    0.90,
                    0.95,
                ],
            )
        )

        scenario_marginal_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "timestamp_utc": timestamp_utc,
                "price_area": price_area,
                "centre_model": centre_model,
                TARGET_COLUMN: actual,
                "scenario_crps": ensemble_crps(
                    values,
                    actual,
                ),
                "coverage_80": float(
                    q10 <= actual <= q90
                ),
                "width_80": q90 - q10,
                "coverage_90": float(
                    q05 <= actual <= q95
                ),
                "width_90": q95 - q05,
                "scenario_median_mae": abs(
                    q50 - actual
                ),
                "scenario_median_bias": (
                    q50 - actual
                ),
            }
        )


scenario_marginal_metrics = pd.DataFrame(
    scenario_marginal_rows
)

scenario_marginal_summary = (
    scenario_marginal_metrics.groupby(
        [
            "panel",
            "fold_id",
            "price_area",
            "centre_model",
        ],
        as_index=False,
    )
    .agg(
        mean_scenario_crps=(
            "scenario_crps",
            "mean",
        ),
        coverage_80=(
            "coverage_80",
            "mean",
        ),
        width_80=("width_80", "mean"),
        coverage_90=(
            "coverage_90",
            "mean",
        ),
        width_90=("width_90", "mean"),
        scenario_median_mae=(
            "scenario_median_mae",
            "mean",
        ),
        scenario_median_bias=(
            "scenario_median_bias",
            "mean",
        ),
        observations=(
            "timestamp_utc",
            "size",
        ),
    )
    if not scenario_marginal_metrics.empty
    else pd.DataFrame()
)

scenario_marginal_metrics.to_parquet(
    SCENARIO_DIR
    / "notebook_03_scenario_marginal_metrics.parquet",
    index=False,
)

scenario_marginal_summary.to_csv(
    TABLE_DIR
    / "notebook_03_scenario_marginal_summary.csv",
    index=False,
)

display(scenario_marginal_summary)


In [ ]:
# ============================================================
# 22. DAILY JOINT ENERGY SCORE AND TOTAL-WIND COVERAGE
# ============================================================

def approximate_energy_score(
    scenario_matrix: np.ndarray,
    actual_vector: np.ndarray,
    *,
    pair_draws: int,
    seed: int,
) -> float:
    scenarios = np.asarray(
        scenario_matrix,
        dtype=float,
    )

    actual = np.asarray(
        actual_vector,
        dtype=float,
    )

    first_term = float(
        np.mean(
            np.linalg.norm(
                scenarios
                - actual[None, :],
                axis=1,
            )
        )
    )

    rng = np.random.default_rng(seed)

    first_indices = rng.integers(
        0,
        len(scenarios),
        size=pair_draws,
    )

    second_indices = rng.integers(
        0,
        len(scenarios),
        size=pair_draws,
    )

    second_term = float(
        np.mean(
            np.linalg.norm(
                scenarios[first_indices]
                - scenarios[second_indices],
                axis=1,
            )
        )
    )

    return (
        first_term
        - 0.5 * second_term
    )


daily_joint_rows = []
daily_total_rows = []

if not joint_scenarios.empty:
    for keys, day in joint_scenarios.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date_local",
        ]
    ):
        (
            panel_name,
            fold_id,
            delivery_date,
        ) = keys

        actual_wide = (
            day.drop_duplicates(
                [
                    "timestamp_utc",
                    "price_area",
                ]
            )
            .pivot(
                index="timestamp_utc",
                columns="price_area",
                values=TARGET_COLUMN,
            )
            .sort_index()
        )

        scenario_cube = []

        for scenario_id, scenario in day.groupby(
            "scenario_id"
        ):
            scenario_wide = (
                scenario.pivot(
                    index="timestamp_utc",
                    columns="price_area",
                    values="scenario_wind_mwh",
                )
                .reindex(
                    actual_wide.index
                )
            )

            scenario_cube.append(
                scenario_wide[
                    ["DK1", "DK2"]
                ]
                .to_numpy(dtype=float)
                .reshape(-1)
            )

        scenario_matrix = np.vstack(
            scenario_cube
        )

        actual_vector = (
            actual_wide[
                ["DK1", "DK2"]
            ]
            .to_numpy(dtype=float)
            .reshape(-1)
        )

        daily_joint_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "delivery_date_local": delivery_date,
                "joint_energy_score": (
                    approximate_energy_score(
                        scenario_matrix,
                        actual_vector,
                        pair_draws=(
                            ENERGY_SCORE_PAIR_DRAWS
                        ),
                        seed=stable_seed(
                            RANDOM_SEED,
                            panel_name,
                            fold_id,
                            delivery_date,
                            "energy_score",
                        ),
                    )
                ),
                "hours": len(actual_wide),
                "scenario_count": len(
                    scenario_matrix
                ),
            }
        )

        for price_area in (
            "DK1",
            "DK2",
        ):
            area_day = day.loc[
                day["price_area"].eq(
                    price_area
                )
            ]

            actual_total = float(
                area_day.drop_duplicates(
                    "timestamp_utc"
                )[TARGET_COLUMN].sum()
            )

            scenario_totals = (
                area_day.groupby(
                    "scenario_id"
                )["scenario_wind_mwh"]
                .sum()
                .to_numpy(dtype=float)
            )

            q05, q10, q50, q90, q95 = (
                np.quantile(
                    scenario_totals,
                    [
                        0.05,
                        0.10,
                        0.50,
                        0.90,
                        0.95,
                    ],
                )
            )

            daily_total_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold_id,
                    "delivery_date_local": delivery_date,
                    "price_area": price_area,
                    "actual_daily_total_mwh": actual_total,
                    "scenario_daily_total_median_mwh": q50,
                    "daily_total_crps": ensemble_crps(
                        scenario_totals,
                        actual_total,
                    ),
                    "daily_total_coverage_80": float(
                        q10
                        <= actual_total
                        <= q90
                    ),
                    "daily_total_width_80": (
                        q90 - q10
                    ),
                    "daily_total_coverage_90": float(
                        q05
                        <= actual_total
                        <= q95
                    ),
                    "daily_total_width_90": (
                        q95 - q05
                    ),
                }
            )


daily_joint_metrics = pd.DataFrame(
    daily_joint_rows
)

daily_total_metrics = pd.DataFrame(
    daily_total_rows
)

daily_joint_summary = (
    daily_joint_metrics.groupby(
        ["panel", "fold_id"],
        as_index=False,
    )
    .agg(
        mean_joint_energy_score=(
            "joint_energy_score",
            "mean",
        ),
        days=(
            "delivery_date_local",
            "nunique",
        ),
    )
    if not daily_joint_metrics.empty
    else pd.DataFrame()
)

daily_total_summary = (
    daily_total_metrics.groupby(
        [
            "panel",
            "fold_id",
            "price_area",
        ],
        as_index=False,
    )
    .agg(
        mean_daily_total_crps=(
            "daily_total_crps",
            "mean",
        ),
        daily_total_coverage_80=(
            "daily_total_coverage_80",
            "mean",
        ),
        daily_total_width_80=(
            "daily_total_width_80",
            "mean",
        ),
        daily_total_coverage_90=(
            "daily_total_coverage_90",
            "mean",
        ),
        daily_total_width_90=(
            "daily_total_width_90",
            "mean",
        ),
        days=(
            "delivery_date_local",
            "nunique",
        ),
    )
    if not daily_total_metrics.empty
    else pd.DataFrame()
)

daily_joint_metrics.to_csv(
    TABLE_DIR
    / "notebook_03_daily_joint_energy_scores.csv",
    index=False,
)

daily_total_metrics.to_csv(
    TABLE_DIR
    / "notebook_03_daily_total_scenario_metrics.csv",
    index=False,
)

daily_joint_summary.to_csv(
    TABLE_DIR
    / "notebook_03_daily_joint_energy_score_summary.csv",
    index=False,
)

daily_total_summary.to_csv(
    TABLE_DIR
    / "notebook_03_daily_total_scenario_summary.csv",
    index=False,
)

display(daily_joint_summary)
display(daily_total_summary)


In [ ]:
# ============================================================
# 23. SCENARIO DEPENDENCE DIAGNOSTICS
# ============================================================

dependence_rows = []

if not joint_scenarios.empty:
    for keys, group in joint_scenarios.groupby(
        ["panel", "fold_id"]
    ):
        panel_name, fold_id = keys

        actual_residual = (
            primary_centre_predictions.loc[
                primary_centre_predictions[
                    "panel"
                ].eq(panel_name)
                & primary_centre_predictions[
                    "fold_id"
                ].eq(fold_id)
            ]
            .pivot(
                index="timestamp_utc",
                columns="price_area",
                values="residual",
            )
            .dropna()
        )

        actual_correlation = float(
            actual_residual["DK1"].corr(
                actual_residual["DK2"]
            )
        )

        scenario_correlations = []

        for scenario_id, scenario in group.groupby(
            "scenario_id"
        ):
            residual_wide = (
                scenario.assign(
                    scenario_residual=lambda frame: (
                        frame["scenario_wind_mwh"]
                        - frame["centre_prediction"]
                    )
                )
                .pivot(
                    index="timestamp_utc",
                    columns="price_area",
                    values="scenario_residual",
                )
                .dropna()
            )

            if len(residual_wide) >= 2:
                scenario_correlations.append(
                    float(
                        residual_wide[
                            "DK1"
                        ].corr(
                            residual_wide[
                                "DK2"
                            ]
                        )
                    )
                )

        dependence_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "actual_residual_correlation": actual_correlation,
                "mean_scenario_residual_correlation": float(
                    np.nanmean(
                        scenario_correlations
                    )
                ),
                "std_scenario_residual_correlation": float(
                    np.nanstd(
                        scenario_correlations
                    )
                ),
                "scenario_count": len(
                    scenario_correlations
                ),
            }
        )


scenario_dependence = pd.DataFrame(
    dependence_rows
)

scenario_dependence.to_csv(
    TABLE_DIR
    / "notebook_03_scenario_dependence_diagnostics.csv",
    index=False,
)

display(scenario_dependence)


In [ ]:
# ============================================================
# 24. EXAMPLE INTERVAL AND SCENARIO FIGURES
# ============================================================

for (
    panel_name,
    price_area,
), centre_model in PRIMARY_CENTRES.items():
    available_methods = (
        probabilistic_summary.loc[
            probabilistic_summary["panel"].eq(
                panel_name
            )
            & probabilistic_summary[
                "price_area"
            ].eq(price_area)
            & probabilistic_summary[
                "centre_model"
            ].eq(centre_model)
        ]
        .sort_values("mean_wis")
    )

    if available_methods.empty:
        continue

    best_method = str(
        available_methods.iloc[0][
            "probabilistic_method"
        ]
    )

    selected = quantile_wide.loc[
        quantile_wide["panel"].eq(
            panel_name
        )
        & quantile_wide[
            "price_area"
        ].eq(price_area)
        & quantile_wide[
            "centre_model"
        ].eq(centre_model)
        & quantile_wide[
            "probabilistic_method"
        ].eq(best_method)
    ].sort_values("timestamp_utc")

    if selected.empty:
        continue

    example = selected.head(
        7 * 24
    )

    plt.figure(figsize=(13, 6))
    plt.fill_between(
        example["timestamp_utc"],
        example[quantile_column(0.05)],
        example[quantile_column(0.95)],
        alpha=0.20,
        label="90% interval",
    )
    plt.fill_between(
        example["timestamp_utc"],
        example[quantile_column(0.10)],
        example[quantile_column(0.90)],
        alpha=0.30,
        label="80% interval",
    )
    plt.plot(
        example["timestamp_utc"],
        example[TARGET_COLUMN],
        label="actual",
    )
    plt.plot(
        example["timestamp_utc"],
        example[quantile_column(0.50)],
        label="median",
    )
    plt.title(
        f"{panel_name.upper()} {price_area}: "
        f"{centre_model} + {best_method}"
    )
    plt.xlabel("Delivery time UTC")
    plt.ylabel("Wind MWh")
    plt.legend()

    save_current_figure(
        f"notebook_03_interval_example_"
        f"{panel_name}_{price_area}.png"
    )


if not joint_scenarios.empty:
    example_keys = (
        joint_scenarios[
            [
                "panel",
                "fold_id",
                "delivery_date_local",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "panel",
                "fold_id",
                "delivery_date_local",
            ]
        )
        .iloc[0]
    )

    example_day = joint_scenarios.loc[
        joint_scenarios["panel"].eq(
            example_keys["panel"]
        )
        & joint_scenarios[
            "fold_id"
        ].eq(example_keys["fold_id"])
        & joint_scenarios[
            "delivery_date_local"
        ].eq(
            example_keys[
                "delivery_date_local"
            ]
        )
    ]

    for price_area in (
        "DK1",
        "DK2",
    ):
        area_day = example_day.loc[
            example_day["price_area"].eq(
                price_area
            )
        ]

        scenario_quantiles = (
            area_day.groupby(
                "timestamp_utc"
            )["scenario_wind_mwh"]
            .quantile(
                [
                    0.05,
                    0.10,
                    0.50,
                    0.90,
                    0.95,
                ]
            )
            .unstack()
            .sort_index()
        )

        actual = (
            area_day.drop_duplicates(
                "timestamp_utc"
            )
            .set_index(
                "timestamp_utc"
            )[TARGET_COLUMN]
            .sort_index()
        )

        plt.figure(figsize=(13, 6))
        plt.fill_between(
            scenario_quantiles.index,
            scenario_quantiles[0.05],
            scenario_quantiles[0.95],
            alpha=0.20,
            label="scenario 90% interval",
        )
        plt.fill_between(
            scenario_quantiles.index,
            scenario_quantiles[0.10],
            scenario_quantiles[0.90],
            alpha=0.30,
            label="scenario 80% interval",
        )
        plt.plot(
            actual.index,
            actual.values,
            label="actual",
        )
        plt.plot(
            scenario_quantiles.index,
            scenario_quantiles[0.50],
            label="scenario median",
        )
        plt.title(
            f"{example_keys['panel'].upper()} "
            f"{price_area}: joint scenario example"
        )
        plt.xlabel("Delivery time UTC")
        plt.ylabel("Wind MWh")
        plt.legend()

        save_current_figure(
            f"notebook_03_scenario_example_"
            f"{example_keys['panel']}_"
            f"{price_area}.png"
        )


In [ ]:
# ============================================================
# 25. FINAL VERIFICATION
# ============================================================

blocking_failures: list[str] = []

if notebook_02_verification.get(
    "verification_status"
) != "MODELLING_COMPLETE":
    blocking_failures.append(
        "Notebook 02 was not verified complete."
    )

if calibration_manifest.empty:
    blocking_failures.append(
        "Calibration manifest is empty."
    )

if (
    calibration_manifest[
        "history_last_timestamp_utc"
    ]
    >= calibration_manifest[
        "evaluation_first_timestamp_utc"
    ]
).any():
    blocking_failures.append(
        "At least one probabilistic calibration task "
        "uses overlapping or future residual history."
    )

if probabilistic_predictions.empty:
    blocking_failures.append(
        "Probabilistic predictions are empty."
    )

expected_quantiles = set(
    QUANTILE_LEVELS
)

observed_quantiles = set(
    probabilistic_predictions[
        "quantile"
    ].unique()
)

if observed_quantiles != expected_quantiles:
    blocking_failures.append(
        "The output quantile grid is incomplete."
    )

if (
    probabilistic_predictions[
        "probabilistic_prediction"
    ] < 0
).any():
    blocking_failures.append(
        "Negative wind quantile predictions remain."
    )

crossing_check = (
    probabilistic_predictions.pivot(
        index=[
            "panel",
            "fold_id",
            "timestamp_utc",
            "price_area",
            "centre_model",
            "probabilistic_method",
        ],
        columns="quantile",
        values="probabilistic_prediction",
    )
    .sort_index(axis=1)
    .diff(axis=1)
)

if (
    crossing_check.iloc[:, 1:]
    < -1e-9
).any().any():
    blocking_failures.append(
        "Quantile crossing remains after repair."
    )

required_panel_area_pairs = {
    ("d2", "DK1"),
    ("d2", "DK2"),
    ("exact", "DK1"),
    ("exact", "DK2"),
}

observed_panel_area_pairs = set(
    probabilistic_predictions[
        ["panel", "price_area"]
    ]
    .drop_duplicates()
    .itertuples(
        index=False,
        name=None,
    )
)

if (
    observed_panel_area_pairs
    != required_panel_area_pairs
):
    blocking_failures.append(
        "Probabilistic outputs do not cover all "
        "four panel-area combinations."
    )

if RUN_SCENARIOS:
    if joint_scenarios.empty:
        blocking_failures.append(
            "Scenario generation was enabled but "
            "joint scenarios are empty."
        )
    else:
        scenario_counts = (
            joint_scenarios.groupby(
                [
                    "panel",
                    "fold_id",
                    "timestamp_utc",
                    "price_area",
                ]
            )["scenario_id"]
            .nunique()
        )

        if not scenario_counts.eq(
            SCENARIO_COUNT
        ).all():
            blocking_failures.append(
                "At least one timestamp does not have "
                "the configured scenario count."
            )

        if (
            joint_scenarios[
                "scenario_wind_mwh"
            ] < 0
        ).any():
            blocking_failures.append(
                "Negative scenario wind values remain."
            )

verification_status = (
    "PROBABILISTIC_MODELLING_COMPLETE"
    if not blocking_failures
    else "BLOCKED"
)

verification = {
    "verification_status": verification_status,
    "blocking_failures": blocking_failures,
    "input_manifest": input_manifest,
    "dropped_missing_prediction_rows": int(
        dropped_missing_prediction_rows
    ),
    "models_with_missing_predictions": int(
        missing_prediction_audit[
            "missing_predictions"
        ].gt(0).sum()
    ),
    "candidate_centres": {
        f"{panel}/{area}": list(models)
        for (
            panel,
            area,
        ), models in CANDIDATE_CENTRES.items()
    },
    "primary_centres": {
        f"{panel}/{area}": model
        for (
            panel,
            area,
        ), model in PRIMARY_CENTRES.items()
    },
    "calibration_tasks": int(
        len(calibration_manifest)
    ),
    "probabilistic_prediction_rows": int(
        len(probabilistic_predictions)
    ),
    "probabilistic_methods": sorted(
        probabilistic_predictions[
            "probabilistic_method"
        ].unique()
    ),
    "quantile_levels": list(
        QUANTILE_LEVELS
    ),
    "probabilistic_panel_area_pairs": sorted(
        [
            list(pair)
            for pair in observed_panel_area_pairs
        ]
    ),
    "scenario_generation_enabled": (
        RUN_SCENARIOS
    ),
    "scenario_count_per_timestamp": (
        SCENARIO_COUNT
        if RUN_SCENARIOS
        else 0
    ),
    "joint_scenario_rows": int(
        len(joint_scenarios)
    ),
    "scenario_panel_area_pairs": (
        sorted(
            [
                list(pair)
                for pair in set(
                    joint_scenarios[
                        [
                            "panel",
                            "price_area",
                        ]
                    ]
                    .drop_duplicates()
                    .itertuples(
                        index=False,
                        name=None,
                    )
                )
            ]
        )
        if not joint_scenarios.empty
        else []
    ),
}

verification_path = (
    AUDIT_DIR
    / "notebook_03_verification.json"
)

with verification_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        verification,
        file,
        indent=2,
        default=str,
    )

print("=== NOTEBOOK 03 VERIFICATION ===")
print(
    json.dumps(
        verification,
        indent=2,
        default=str,
    )
)

if blocking_failures:
    raise RuntimeError(
        "Notebook 03 verification failed: "
        f"{blocking_failures}"
    )


## Outputs

Notebook 03 writes to:

```text
outputs/notebook_03_probabilistic/
```

Principal artifacts:

```text
audits/notebook_03_verification.json
audits/notebook_03_calibration_manifest.csv
audits/notebook_03_input_missing_prediction_audit.csv
audits/notebook_03_centre_availability.csv

predictions/notebook_03_quantile_predictions.parquet
predictions/notebook_03_probabilistic_row_metrics.parquet
predictions/notebook_03_pit_values.parquet
predictions/notebook_03_primary_centre_predictions.parquet
predictions/notebook_03_provider_disagreement_features.parquet

tables/notebook_03_probabilistic_metrics_by_fold.csv
tables/notebook_03_probabilistic_metric_summary.csv
tables/notebook_03_quantile_calibration_by_fold.csv
tables/notebook_03_quantile_reliability.csv
tables/notebook_03_pit_summary.csv
tables/notebook_03_conformal_adjustments.csv

scenarios/notebook_03_joint_wind_scenarios.parquet
scenarios/notebook_03_scenario_marginal_metrics.parquet

tables/notebook_03_scenario_marginal_summary.csv
tables/notebook_03_daily_joint_energy_scores.csv
tables/notebook_03_daily_joint_energy_score_summary.csv
tables/notebook_03_daily_total_scenario_metrics.csv
tables/notebook_03_daily_total_scenario_summary.csv
tables/notebook_03_scenario_dependence_diagnostics.csv
```

The scenario parquet is the principal input for the later nomination and
imbalance-settlement notebook.
